# 1. Data preparation

Builds one table with one row per loan from the raw Freddie Mac files.

**Input:** raw origination and performance files (2005–2009).  
**Output:** `data/loan_data.pkl`.

**1.1 Dataset description  
1.2 Load the origination files  
1.3 Load and summarise the performance files  
1.4 Merge the two tables  
1.5 Checks  
1.6 Replace the "not available" codes with NaN  
1.7 Save**

## 1.1 Dataset description

### Source

Freddie Mac Single-Family Loan-Level Dataset (SFLLD), sample files for the vintages 2005–2009
(50,000 loans per vintage). The data was downloaded from Freddie Mac's Clarity portal
(registration required) and is not included in this repository.

### File format

For each vintage there are two files:
- `sample_orig_YYYY.txt`: origination file, one row per loan (loan characteristics at origination).
- `sample_perf_YYYY.txt`: performance file, one row per loan per month (monthly history of the loan).

Both are pipe-delimited (`|`) text files **without a header row**, and they are linked through `loan_id`.
The column names used in this project are **our own** abbreviations of the official field names
in Freddie Mac's *General User Guide* (July 2026 edition). All amounts are in US dollars and all dates use the `YYYYMM` format.

### Origination file: `sample_orig_YYYY.txt`

One row per loan, 31 columns. It describes the loan and the borrower **at the time the loan was originated** and does not change over time.

| # | Column | Official name | What it measures | Format / codes |
|---|---|---|---|---|
| 1 | `credit_score` | Classic FICO | Borrower's credit score at origination | 300–850; `9999` = not available |
| 2 | `first_payment_date` | First Payment Date | Month of the first scheduled payment | YYYYMM |
| 3 | `first_time_homebuyer` | First Time Homebuyer Indicator | Whether the borrower is a first-time homebuyer | `Y` yes, `N` no, `9` not available |
| 4 | `maturity_date` | Maturity Date | Month of the final scheduled payment | YYYYMM |
| 5 | `msa` | Metropolitan Statistical Area | Code of the metropolitan area of the property | 5 digits; blank = outside an MSA or unknown |
| 6 | `mi_pct` | Mortgage Insurance Percentage | Percentage of mortgage insurance coverage | 1–55 (%); `0` = no insurance; `999` = not available |
| 7 | `num_units` | Number of Units | Number of dwelling units in the property | 1–4; `99` = not available |
| 8 | `occupancy_status` | Occupancy Status | How the property is used | `P` primary residence, `I` investment, `S` second home, `9` not available |
| 9 | `cltv` | Original Combined Loan-to-Value | All known loans secured by the property divided by the property value | %; `999` = not available |
| 10 | `dti` | Original Debt-to-Income Ratio | Borrower's monthly debt payments divided by monthly income | 0–65 (%); `999` = not available |
| 11 | `orig_upb` | Original UPB | Original loan amount (UPB = Unpaid Principal Balance) | $, rounded to the nearest $1,000 |
| 12 | `ltv` | Original Loan-to-Value | Loan amount divided by the property value | %; `999` = not available |
| 13 | `int_rate` | Original Interest Rate | Interest rate on the note | % |
| 14 | `channel` | Channel | How the loan was originated | `R` retail, `B` broker, `C` correspondent, `T` third party, `9` not available |
| 15 | `prepay_penalty` | Prepayment Penalty Indicator | Whether early repayment is penalised | `Y` / `N` |
| 16 | `amortization_type` | Amortization Type | Type of interest rate | `FRM` fixed rate, `ARM` adjustable rate |
| 17 | `property_state` | Property State | State where the property is located | two letters (e.g. GA, TX) |
| 18 | `property_type` | Property Type | Type of property | `SF` single-family, `CO` condominium, `PU` planned unit development, `CP` cooperative, `MH` manufactured housing, `99` not available |
| 19 | `postal_code` | Postal Code | First three digits of the postal code | `000` = unknown |
| 20 | `loan_id` | Loan Identifier | Unique loan identifier, the same in both files | `PYYQnXXXXXXX`: `P` = `F` (fixed) or `A` (adjustable), `YYQn` = origination year and quarter |
| 21 | `loan_purpose` | Loan Purpose | Purpose of the loan | `P` purchase, `C` cash-out refinance, `N` no-cash-out refinance, `R` refinance (not specified), `9` not available |
| 22 | `orig_loan_term` | Original Loan Term | Length of the loan | months |
| 23 | `num_borrowers` | Number of Borrowers | Number of borrowers | up to 2018Q1: `1` one, `2` more than one; later: actual count; `99` = not available |
| 24 | `seller_name` | Seller Name | Entity that sold the loan to Freddie Mac | text; `OTHER` if the seller has under 1% of the quarter |
| 25 | `super_conforming` | Super Conforming Flag | Whether the loan exceeds the standard conforming loan limit | `Y` / `N`; applies to loans originated from 1 Oct 2008 |
| 26 | `pre_harp_loan_id` | Pre-HARP Loan Sequence Number | For a relief refinance (HARP), the `loan_id` of the refinanced loan | blank unless HARP |
| 27 | `special_eligibility_program` | Special Eligibility Program | Participation in a Freddie Mac program | `H`, `F`, `R`; blank; reported from 2015 |
| 28 | `harp_indicator` | HARP Indicator | Whether the loan is a relief refinance (HARP) | `Y` / `N` |
| 29 | `property_valuation_method` | Property Valuation Method | How the property value was determined | `1`–`4` various methods, `7` = not available; reported from 2017 |
| 30 | `io_indicator` | Interest Only Indicator | Whether only interest is paid for an initial period | `Y` / `N` |
| 31 | `vantagescore_4` | VantageScore 4.0 | Second credit score of the borrower | 300–850; `9999` = not available |

### Performance file: `sample_perf_YYYY.txt`

One row per loan per month, 35 columns, from origination until the loan closes or the data ends (March 2026). It describes **what happens to the loan each month**.

**Sign convention:** recoveries and gains are **negative** amounts. Expenses and losses are **positive**. `actual_loss` is positive for a loss and negative for a gain.

| # | Column | Official name | What it measures | Format / codes |
|---|---|---|---|---|
| 1 | `loan_id` | Loan Identifier | Loan identifier (same as in the origination file) | |
| 2 | `period` | Period | The month the row refers to | YYYYMM |
| 3 | `current_upb` | Current Actual UPB | Outstanding principal balance at the end of the month | $ |
| 4 | `delinquency_status` | Current Loan Delinquency Status | Payment delinquency | `00` current or under 30 days, `01` 30–59, `02` 60–89, `03` 90–119, and so on (capped at 99), `RA` property acquired by the lender (REO), `XX` not available |
| 5 | `loan_age` | Loan Age | Age of the loan | months |
| 6 | `remaining_months` | Remaining Months to Legal Maturity | Months left until maturity | months |
| 7 | `defect_settlement_date` | Underwriting Defect and Major Servicing Defect Settlement Date | When a defect in underwriting or servicing was resolved | YYYYMM; blank if not applicable |
| 8 | `modification_flag` | Modification Flag | Whether the loan was modified | `Y` this month, `P` earlier, blank = no |
| 9 | `zero_balance_code` | Zero Balance Code | Reason the balance went to zero (filled only in the loan's last row) | `01` prepaid or matured, `02` third-party sale, `03` short sale or charge-off, `09` sale of the property after foreclosure (REO), `15` whole loan sale, `16` reperforming loan securitization, `96` underwriting or servicing defect before a credit event |
| 10 | `zero_balance_date` | Zero Balance Effective Date | Month the loan closed | YYYYMM |
| 11 | `current_int_rate` | Current Interest Rate | Interest rate in effect that month | % |
| 12 | `non_int_bearing_upb` | Current Non-Interest Bearing UPB | Part of the balance that bears no interest (deferred) | $ |
| 13 | `ddlpi` | Due Date of Last Paid Installment | Due date of the last installment that was paid | YYYYMM |
| 14 | `mi_recoveries` | MI Recoveries | Proceeds from mortgage insurance | $ (negative) |
| 15 | `net_sale_proceeds` | Net Sales Proceeds | Proceeds from selling the property, after selling expenses | $ (negative) |
| 16 | `non_mi_recoveries` | Non-MI Recoveries | Other recoveries (settlements, tax or insurance refunds, rental income, etc.) | $ (negative) |
| 17 | `total_expenses` | Total Expenses | Costs of acquiring, maintaining and disposing of the property (sum of 18–21) | $ |
| 18 | `legal_costs` | Legal Costs | Legal costs | $ |
| 19 | `maintenance_costs` | Maintenance and Preservation Costs | Maintenance, preservation and repair costs | $ |
| 20 | `taxes_insurance` | Taxes and Insurance | Taxes and insurance owed | $ |
| 21 | `misc_expenses` | Miscellaneous Expenses | Other sale-related costs (title, administrative, auction fees) | $ |
| 22 | `actual_loss` | Actual Loss | Realized loss: `zero_balance_removal_upb` + net sale proceeds + delinquent accrued interest + expenses + recoveries | $; calculated only for codes `02`, `03`, `09`, `15` |
| 23 | `cum_modification_costs` | Cumulative Modification Costs | Running total of modification costs | $ |
| 24 | `int_rate_step_indicator` | Interest Rate Step Indicator | Whether the modification has a step rate | `Y` / `N` / blank |
| 25 | `payment_deferral_flag` | Payment Deferral Flag | Whether a payment deferral was granted | `C` current period, `P` prior period, blank = no |
| 26 | `eltv` | Estimated Loan-to-Value | Balance divided by the estimated current property value | %; `999` = not available; from April 2017 |
| 27 | `zero_balance_removal_upb` | Zero Balance Removal UPB | Balance immediately before the loan closed | $ |
| 28 | `delinquent_accrued_interest` | Delinquent Accrued Interest | Interest owed at default | $; only for codes `02`, `03`, `09`, `15` |
| 29 | `disaster_delinquency` | Delinquency Due to Disaster | Delinquency caused by a natural disaster | `Y` / blank; from January 2014 |
| 30 | `borrower_assistance_plan` | Borrower Assistance Plan | Payment relief plan the borrower is enrolled in | `F` forbearance, `R` repayment, `T` trial period, blank = none; from January 2014 |
| 31 | `current_period_mod_costs` | Current Period Modification Costs | Modification cost for the current month | $ |
| 32 | `int_bearing_upb` | Current Interest Bearing UPB | Part of the balance that bears interest | $ |
| 33 | `mi_cancellation_indicator` | Mortgage Insurance Cancellation Indicator | Whether mortgage insurance was cancelled | `Y` / `N` / `7` = not available; from 2015 |
| 34 | `servicer_name` | Servicer Name | Entity servicing the loan | text; `OTHER` if under 1% of the quarter |
| 35 | `bankruptcy_cramdown_costs` | Bankruptcy Cramdown Costs | Cost of a court-ordered principal reduction in bankruptcy | $; only in the month the loan closes |

## 1.2 Load the origination files

We read the five origination files (2005–2009), add a `vintage` column and stack them into one table. The columns are described in section 1.1.

In [1]:
import numpy as np
import pandas as pd

data_dir = '../data'  # relative to the notebooks folder

In [2]:
orig_cols = ['credit_score','first_payment_date','first_time_homebuyer','maturity_date','msa',
             'mi_pct','num_units','occupancy_status','cltv','dti','orig_upb','ltv','int_rate',
             'channel','prepay_penalty','amortization_type','property_state','property_type',
             'postal_code','loan_id','loan_purpose','orig_loan_term','num_borrowers',
             'seller_name','super_conforming','pre_harp_loan_id','special_eligibility_program',
             'harp_indicator','property_valuation_method','io_indicator','vantagescore_4']
len(orig_cols)

31

In [3]:
year_list = [2005, 2006, 2007, 2008, 2009]
tables = []
for year in year_list:
    path = f'{data_dir}/sample_{year}/sample_orig_{year}.txt'
    table = pd.read_csv(path, sep = '|', header = None, names = orig_cols)
    table['vintage'] = year
    tables.append(table)
orig = pd.concat(tables, ignore_index = True)

orig.shape

(250000, 32)

In [4]:
orig['loan_id'].is_unique
# Check that each loan appears only once

True

In [5]:
orig['vintage'].value_counts()
# Check the number of loans per year

vintage
2005    50000
2006    50000
2007    50000
2008    50000
2009    50000
Name: count, dtype: int64

## 1.3 Load and summarise the performance files

We read the performance files and keep only the last row of each loan that closed, i.e. the row with the closing information. The columns are described in section 1.1.

### What we keep, and why

From the 35 columns we need only these four to build the targets of the three models:

| Column | Why we keep it |
|---|---|
| `loan_id` | Key used to merge the summary with the origination table. |
| `zero_balance_code` | Reason the loan closed. Codes `02`, `03`, `09`, `15` are credit-event closures (third-party sale, short sale or charge-off, REO disposition, whole loan sale). They define our default flag (`default = 1`, which is the target for the PD model) and are the loans used for the LGD and EAD models. |
| `zero_balance_removal_upb` | Unpaid principal balance right before the loan closed. It is the exposure used for EAD and the denominator of LGD. |
| `actual_loss` | Realised loss reported by Freddie Mac. It is the numerator of LGD. |

The other columns are not used. Most describe what happens after origination, so they cannot be model inputs (they would leak future information); the rest are not needed for the targets.

Loans that have not closed yet stay in the final table (section 1.4) with `default = 0`, which is a limitation of this approach.

In [6]:
perf_cols = ['loan_id','period','current_upb','delinquency_status','loan_age','remaining_months',
             'defect_settlement_date','modification_flag','zero_balance_code','zero_balance_date',
             'current_int_rate','non_int_bearing_upb','ddlpi','mi_recoveries','net_sale_proceeds',
             'non_mi_recoveries','total_expenses','legal_costs','maintenance_costs','taxes_insurance',
             'misc_expenses','actual_loss','cum_modification_costs','int_rate_step_indicator',
             'payment_deferral_flag','eltv','zero_balance_removal_upb','delinquent_accrued_interest',
             'disaster_delinquency','borrower_assistance_plan','current_period_mod_costs',
             'int_bearing_upb','mi_cancellation_indicator','servicer_name','bankruptcy_cramdown_costs']
len(perf_cols)
# All columns

35

In [7]:
perf_use = ['loan_id', 'zero_balance_code', 'zero_balance_removal_upb', 'actual_loss']
# The columns we keep

In [8]:
year_list = [2005, 2006, 2007, 2008, 2009]
tables = []
for year in year_list:
    path = f'{data_dir}/sample_{year}/sample_perf_{year}.txt'
    table = pd.read_csv(path, sep = '|', header = None, names = perf_cols, usecols = perf_use, dtype = {'zero_balance_code': str})
    table = table[table['zero_balance_code'].notna()]
    tables.append(table)
perf = pd.concat(tables, ignore_index = True)

perf.shape

(242999, 4)

In [9]:
perf['loan_id'].is_unique
# Check that each loan appears only once

True

In [10]:
perf['zero_balance_code'].value_counts()
# Count number of loans for each code

zero_balance_code
01    223882
09      7808
16      3647
03      3564
02      1917
96      1386
15       795
Name: count, dtype: int64

**Remark.** Two closing codes do not describe a clear credit outcome: `96` (the loan was returned because of an underwriting or servicing defect) and `16` (modified loans that performed again and were securitised). We remove these loans from the final table in section 1.4. This is a limitation: the `16` loans are probably riskier than average, so the estimated default rate may be slightly lower than the true one.

## 1.4 Merge the two tables

We merge the origination table (one row per loan, all 250,000 loans) with the table of closing information (one row per loan that closed), using `loan_id` as the key.

We use a **left merge**, so that every loan in the origination table is kept. Loans that have not closed yet have no closing information, so `zero_balance_code`, `zero_balance_removal_upb` and `actual_loss` are missing (`NaN`) for them.

After the merge we:
1. create the `default` flag: `1` if `zero_balance_code` is `02`, `03`, `09` or `15`, and `0` otherwise.
2. remove the loans closed with code `16` or `96`, which do not describe a clear credit outcome (see section 1.3).

In [11]:
loan_data = orig.merge(perf, on='loan_id', how='left')
loan_data.shape

(250000, 35)

Create the `default` column.

In [12]:
type(loan_data['zero_balance_code'][0])
# The values of zero_balance_code are strings, so we compare with strings in the isin below.

str

In [13]:
loan_data['default'] = np.where(loan_data['zero_balance_code'].isin(['02', '03', '09', '15']), 1, 0)
# Create default flag.

loan_data['default'].sum()

np.int64(14084)

In [14]:
loan_data = loan_data[~loan_data['zero_balance_code'].isin(['16', '96'])]
# Remove loans with `zero_balance_code`: 16, 96.

loan_data.shape

(244967, 36)

## 1.5 Checks

Before saving the table we check that it is consistent:

1. **Default rate** (overall and by vintage): the target of the PD model should have plausible values.
2. **Missing values** in each column: to decide what to do with each column.
3. **Closing information of the defaults**: the LGD and EAD models need `actual_loss` and `zero_balance_removal_upb` for the loans with `default = 1`.

In [15]:
loan_data['default'].mean()
# Default rate overall

np.float64(0.057493458302546874)

In [16]:
loan_data.groupby('vintage')['default'].mean()
# Default rate by vintage

vintage
2005    0.057979
2006    0.083373
2007    0.092727
2008    0.043633
2009    0.010949
Name: default, dtype: float64

**Remark.** The default rate is 5.7% overall. By vintage it rises from 5.8% (2005) to 9.3% (2007) and then falls to 4.4% (2008) and 1.1% (2009). This pattern is consistent with the 2007–2009 housing crisis and the tighter underwriting that followed, but we have not compared it with Freddie Mac's published statistics.

In [17]:
loan_data.isnull().sum()

credit_score                        0
first_payment_date                  0
first_time_homebuyer                0
maturity_date                       0
msa                             47028
mi_pct                              0
num_units                           0
occupancy_status                    0
cltv                                0
dti                                 0
orig_upb                            0
ltv                                 0
int_rate                            0
channel                             0
prepay_penalty                      0
amortization_type                   0
property_state                      0
property_type                       0
postal_code                         0
loan_id                             0
loan_purpose                        0
orig_loan_term                      0
num_borrowers                       0
seller_name                         0
super_conforming                    0
pre_harp_loan_id               239704
special_elig

**Remark:** Counting the missing values (`isnull().sum()`) shows the following:

| Column | Missing | What it means and what we do |
|---|---|---|
| `zero_balance_code`, `zero_balance_removal_upb` | 7,001 | The loan has not closed yet. The information is correct, so we leave it as it is. |
| `actual_loss` | 231,981 | Freddie Mac reports `actual_loss` only for loans closed with a credit event (codes `02`, `03`, `09`, `15`). It is therefore missing for loans that are still open and for loans that closed by prepayment. In addition, 1,098 of the 14,084 loans with `default = 1` have no reported loss (14,084 defaults minus 12,986 loans with a value). The loss of these loans is unknown, so they cannot be used for the LGD model, although they stay in the PD model. We handle this in the LGD notebook. |
| `msa` | 47,028 | The property is outside a metropolitan area, or its location is unknown. We decide how to treat it when we prepare the PD model. |
| `pre_harp_loan_id` | 239,704 | Filled only for relief refinance (HARP) loans, where it contains the `loan_id` of the loan that was refinanced (5,263 loans). It is an identifier, not a characteristic of the loan or the borrower, so it will not be a model input. `harp_indicator` already tells whether a loan is a HARP loan. |
| `special_eligibility_program` | 244,967 (all) | Only reported from 2015 onwards, so it is empty for the 2005–2009 vintages. We drop this column. |

The other columns show no missing values, but this is misleading: in those columns "not available" is coded with special values (for example `9999` in `credit_score` or `999` in `ltv` and `dti`). We replace these codes with missing values in the next section.

In [18]:
loan_data = loan_data.drop(columns = ['special_eligibility_program'])
loan_data.shape

(244967, 35)

In [19]:
loan_data[(loan_data['default'] == 1) & (loan_data['actual_loss'].isnull())].shape[0]
# Defaults without a reported loss

1098

**Remark.** 1,098 of the 14,084 loans with `default = 1` have no reported `actual_loss`. They cannot be used for the LGD model (12,986 defaults remain), but they stay in the PD model.

In [20]:
loan_data[(loan_data['default'] == 1) & (loan_data['zero_balance_removal_upb'].isnull())].shape[0]
# Defaults without a closing balance

0

**Remark.** All loans with `default = 1` have `zero_balance_removal_upb`, so all 14,084 defaults can be used for the EAD model.

## 1.6 Replace the "not available" codes with NaN

Some columns code "not available" with a special value instead of leaving the field empty. A model would read these values as real numbers (e.g. a credit score of 9999), so we replace them with `NaN`, following the data dictionary (section 1.1):

| Column | "Not available" code |
|---|---|
| `credit_score`, `vantagescore_4` | `9999` |
| `mi_pct`, `cltv`, `dti`, `ltv` | `999` |
| `num_units`, `num_borrowers` | `99` |
| `first_time_homebuyer`, `occupancy_status`, `channel`, `loan_purpose` | `9` |
| `property_type` | `99` |
| `property_valuation_method` | `7` |
| `postal_code` | `000` |

We check each column and drop those that contain only the "not available" code, since they carry no information. In the others we replace the code with `NaN`.

In [21]:
loan_data['credit_score'].nunique()

364

In [22]:
loan_data[loan_data['credit_score'] == 9999].shape[0]

155

**Remark.** The not-available code `9999` appears in 155 rows. The column has many other values, so we keep it and replace the code with `NaN`.

In [23]:
loan_data['credit_score'] = loan_data['credit_score'].replace(9999, np.nan)
loan_data['credit_score'].isnull().sum()

np.int64(155)

---

In [24]:
loan_data['ltv'].nunique()

120

In [25]:
loan_data[loan_data['ltv'] == 999].shape[0]

15

**Remark.** The not-available code `999` appears in 15 rows. The column has many other values, so we keep it and replace the code with `NaN`.

In [26]:
loan_data['ltv'] = loan_data['ltv'].replace(999, np.nan)
loan_data['ltv'].isnull().sum()

np.int64(15)

---

In [27]:
loan_data['dti'].nunique()

66

In [28]:
loan_data[loan_data['dti'] == 999].shape[0]

9975

**Remark.** The not-available code `999` appears in 9,975 rows. The column has many other values, so we keep it and replace the code with `NaN`.

In [29]:
loan_data['dti'] = loan_data['dti'].replace(999, np.nan)
loan_data['dti'].isnull().sum()

np.int64(9975)

---

In [30]:
loan_data['vantagescore_4'].nunique()

1

In [31]:
loan_data['vantagescore_4'].unique()

array([9999])

**Remark.** The column `vantagescore_4` contains only one value, the not-available code `9999`. It carries no information, so we drop it.

In [32]:
loan_data = loan_data.drop(columns = 'vantagescore_4')
loan_data.shape

(244967, 34)

---

In [33]:
loan_data['property_valuation_method'].nunique()

1

In [34]:
loan_data['property_valuation_method'].unique()

array([7])

**Remark.** The column `property_valuation_method` contains only one value, the not-available code `7`. It carries no information, so we drop it.

In [35]:
loan_data = loan_data.drop(columns = 'property_valuation_method')
loan_data.shape

(244967, 33)

---

In [36]:
loan_data['cltv'].nunique()

152

In [37]:
loan_data[loan_data['cltv'] == 999].shape[0]

16

**Remark.** The not-available code `999` appears in 16 rows. The column has many other values, so we keep it and replace the code with `NaN`.

In [38]:
loan_data['cltv'] = loan_data['cltv'].replace(999, np.nan)
loan_data['cltv'].isnull().sum()

np.int64(16)

---

In [39]:
loan_data['mi_pct'].nunique()

29

In [40]:
loan_data[loan_data['mi_pct'] == 999].shape[0]

2

**Remark.** The not-available code `999` appears in 2 rows. The column has many other values, so we keep it and replace the code with `NaN`. The value `0` means no mortgage insurance and is a real value, so we leave it unchanged.

In [41]:
loan_data['mi_pct'] = loan_data['mi_pct'].replace(999, np.nan)
loan_data['mi_pct'].isnull().sum()

np.int64(2)

---

In [42]:
for col in ['num_units', 'num_borrowers']:
    print(col, loan_data[col].nunique(), (loan_data[col] == 99).sum())

for col in ['first_time_homebuyer', 'occupancy_status', 'channel', 'loan_purpose', 'property_type']:
    print(loan_data[col].value_counts(), '\n')

print(loan_data['postal_code'].dtype)
print(loan_data['postal_code'].value_counts().head())


num_units 5 1
num_borrowers 3 84
first_time_homebuyer
N    220985
Y     23929
9        53
Name: count, dtype: int64 

occupancy_status
P    219828
I     12798
S     12341
Name: count, dtype: int64 

channel
R    119222
T     89536
C     22518
B     13691
Name: count, dtype: int64 

loan_purpose
P    94528
C    83275
N    67164
Name: count, dtype: int64 

property_type
SF    184947
PU     38515
CO     18416
MH      2271
CP       817
99         1
Name: count, dtype: int64 

int64
postal_code
600    2181
606    2100
300    2054
601    1965
750    1893
Name: count, dtype: int64


**Remark.** The not-available codes appear in few rows: `99` in `num_units` (1 row) and in `num_borrowers` (84 rows), `9` in `first_time_homebuyer` (53 rows) and `99` in `property_type` (1 row). These columns have other values, so we keep them and replace the codes with `NaN`. The columns `occupancy_status`, `channel` and `loan_purpose` do not contain the code, so nothing changes.

In [43]:
loan_data['num_units'] = loan_data['num_units'].replace(99, np.nan)
loan_data['num_borrowers'] = loan_data['num_borrowers'].replace(99, np.nan)
loan_data['first_time_homebuyer'] = loan_data['first_time_homebuyer'].replace('9', np.nan)
loan_data['property_type'] = loan_data['property_type'].replace('99', np.nan)

loan_data[['num_units', 'num_borrowers', 'first_time_homebuyer', 'property_type']].isnull().sum()

num_units                1
num_borrowers           84
first_time_homebuyer    53
property_type            1
dtype: int64

---

In [44]:
(loan_data['postal_code'] == 0).sum()

np.int64(1)

**Remark.** The not-available code `000` appears in 1 row. Since `postal_code` was read as a number, the code appears as `0`. The column has many other values, so we keep it and replace the code with `NaN`.

In [45]:
loan_data['postal_code'] = loan_data['postal_code'].replace(0, np.nan)
loan_data['postal_code'].isnull().sum()

np.int64(1)

---

In [46]:
loan_data.isnull().sum()
# Missing values per column after replacing the codes

credit_score                   155
first_payment_date               0
first_time_homebuyer            53
maturity_date                    0
msa                          47028
mi_pct                           2
num_units                        1
occupancy_status                 0
cltv                            16
dti                           9975
orig_upb                         0
ltv                             15
int_rate                         0
channel                          0
prepay_penalty                   0
amortization_type                0
property_state                   0
property_type                    1
postal_code                      1
loan_id                          0
loan_purpose                     0
orig_loan_term                   0
num_borrowers                   84
seller_name                      0
super_conforming                 0
pre_harp_loan_id            239704
harp_indicator                   0
io_indicator                     0
vintage             

In [47]:
loan_data.nunique()
# Check for other columns with only one value

credit_score                   363
first_payment_date              88
first_time_homebuyer             2
maturity_date                  354
msa                            451
mi_pct                          28
num_units                        4
occupancy_status                 3
cltv                           151
dti                             65
orig_upb                       688
ltv                            119
int_rate                       575
channel                          4
prepay_penalty                   2
amortization_type                1
property_state                  54
property_type                    5
postal_code                    888
loan_id                     244967
loan_purpose                     3
orig_loan_term                 168
num_borrowers                    2
seller_name                     27
super_conforming                 2
pre_harp_loan_id              5263
harp_indicator                   2
io_indicator                     1
vintage             

In [48]:
loan_data['amortization_type'].unique()

array(['FRM'], dtype=object)

In [49]:
loan_data['io_indicator'].unique()

array(['N'], dtype=object)

**Remark.** The columns `amortization_type` and `io_indicator` contain only one value (`FRM` and `N`). They carry no information, so we drop them.

In [50]:
loan_data = loan_data.drop(columns = ['amortization_type', 'io_indicator'])
loan_data.shape

(244967, 31)

## 1.7 Save

We save `loan_data` as a pickle file, which keeps the column types and `NaN` values exactly as they are. The next notebooks read this file instead of rerunning the preparation. The file is not uploaded to GitHub; it is recreated by running this notebook.

In [51]:
loan_data.to_pickle(f'{data_dir}/loan_data.pkl')

In [52]:
check = pd.read_pickle(f'{data_dir}/loan_data.pkl')
check.shape

(244967, 31)